In [ ]:
#@title 1. Settings
PROJECT_DIR     = "" #@param {type:"string"}
#@markdown The project folder. Left empty it falls back to DRIVE_ROOT and
#@markdown PROJECT_FOLDER below.
DRIVE_ROOT      = "/content/drive/MyDrive"  #@param {type:"string"}
PROJECT_FOLDER  = "KYNU truncated isoform" #@param {type:"string"}
MODELS_ROOT     = "" #@param {type:"string"}
#@markdown Where the AlphaFold3 results were unpacked. Left empty it searches the
#@markdown project's own md/inputs/af3 and the whole of data/af3, which covers the
#@markdown dated folder the AlphaFold Server download unpacks to. A model file is any
#@markdown .cif or .pdb below a root whose path carries the job name, with a single
#@markdown and a double underscore read as the same thing, since the server collapses
#@markdown them in its folder names.
REF_PDB         = "2HZP" #@param {type:"string"}
#@markdown The crystal structure the cofactor pose is copied from. It must be one of
#@markdown the entries the project's own cofactor measurement used, and the spec
#@markdown records which was taken.
MAX_SUPERPOSE_RMSD_A = 3.0 #@param {type:"number"}
#@markdown The superposition of the crystal subunit onto the model's canonical chain,
#@markdown over the retained block, must be at least this good before the cofactor is
#@markdown copied across.
MAX_ALDIMINE_A  = 2.1 #@param {type:"number"}
#@markdown The cofactor's C4A to lysine NZ distance in the crystal has to be at most
#@markdown this for the pair to be read as an internal aldimine. The lysine found this
#@markdown way is the one CHARMM-GUI substitutes, so it is measured and not typed.
REF_T_K         = 303.15 #@param {type:"number"}
#@markdown The thermostat temperature to set on CHARMM-GUI's dynamics page. Parts 2
#@markdown and 3 read this from the spec and refuse a download built at another value.

import os
PROJECT = PROJECT_DIR.strip() or os.path.join(DRIVE_ROOT, PROJECT_FOLDER)
MD = os.path.join(PROJECT, "md")
INPUTS = os.path.join(MD, "inputs")
RESULTS = os.path.join(MD, "results")
DATA = os.path.join(PROJECT, "data")
STRUCTURES = os.path.join(PROJECT, "structures")
_seen = [os.path.join(d, "build_spec.json") for d, _sub, files in os.walk(MD)
         if "build_spec.json" in files] if os.path.isdir(MD) else []
if len(_seen) == 1:
    INPUTS = os.path.dirname(_seen[0])
elif len(_seen) > 1:
    raise SystemExit("STOP: %d build spec files under %s, %s. Delete the ones that "
                     "are not current, so this run leaves exactly one and part 2 has "
                     "nothing to choose between." % (len(_seen), MD, sorted(_seen)))
CHARMM = os.path.join(INPUTS, "charmmgui")
BUILD_ORDER = ["canonical", "mixed"]
print("project", PROJECT)
print("inputs ", INPUTS)


In [ ]:
#@title 2. Run all starts here: read the models, place the cofactor, write the inputs
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
try:
    import gemmi
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gemmi"], check=True)
    import gemmi

import collections, datetime, glob, json, re

for d in (INPUTS, RESULTS, CHARMM):
    os.makedirs(d, exist_ok=True)


def fatal(msg):
    raise SystemExit("STOP: " + msg)


POCKET_PATH = os.path.join(PROJECT, "results", "cofactor_pocket.json")
PARTITION_PATH = os.path.join(PROJECT, "results", "interface_partition.json")
PROVENANCE = os.path.join(DATA, "construct_provenance.tsv")
PAIRINGS = os.path.join(DATA, "pairings.tsv")
for p in (POCKET_PATH, PARTITION_PATH, PROVENANCE, PAIRINGS):
    if not os.path.isfile(p):
        fatal("no %s. This notebook builds its spec from the project's own result "
              "files, so the project folder has to be the one the analysis ran in."
              % p)
POCKET = json.load(open(POCKET_PATH))["entries"]
KEY = "KYNU " + REF_PDB
if KEY not in POCKET:
    fatal("%s holds no entry for %s, so the cofactor pose would come from a structure "
          "the project never measured. Entries present: %s"
          % (POCKET_PATH, KEY, sorted(POCKET)))
REF = POCKET[KEY]
COFACTOR = REF["cofactor"]
RETAINED_END = REF["retained_end"]
LOST_FROM, LOST_TO = (int(x) for x in REF["lost_block"].split("-"))
CRYSTAL = {k.split()[1]: dict(whole_partner=v["buried_by_whole_partner"],
                              retained_block=v["buried_by_retained_block"],
                              lost_block=v["buried_by_lost_block"])
           for k, v in POCKET.items() if k.startswith("KYNU ")}

LENGTH = {}
for line in open(PROVENANCE).read().strip().split("\n")[1:]:
    name, length, _prov = line.split("\t", 2)
    LENGTH[name] = int(length)
JOBS = {}
for line in open(PAIRINGS).read().strip().split("\n")[1:]:
    job, a, b, role = line.split("\t", 3)
    JOBS[job] = (a, b)
BUILD_JOB = {"canonical": "canonical__canonical", "mixed": "canonical__isoform2"}
for b, j in BUILD_JOB.items():
    if j not in JOBS:
        fatal("pairings.tsv has no job %s, so the %s build has no model to read" % (j, b))
CHAIN_LENGTHS = {b: [LENGTH[c] for c in JOBS[BUILD_JOB[b]]] for b in BUILD_ORDER}
COFACTOR_COPIES = {"canonical": 2, "mixed": 1}

print("cofactor %s, retained block 1 to %d, lost block %d to %d"
      % (COFACTOR, RETAINED_END, LOST_FROM, LOST_TO))
print("chain lengths expected: %s" % CHAIN_LENGTHS)
print("crystal partner burial of the cofactor:")
for e in sorted(CRYSTAL):
    print("   %-6s whole partner %6.2f A2, retained block %6.2f, lost block %6.2f"
          % (e, CRYSTAL[e]["whole_partner"], CRYSTAL[e]["retained_block"],
             CRYSTAL[e]["lost_block"]))

ROOTS = [MODELS_ROOT.strip()] if MODELS_ROOT.strip() else [
    os.path.join(INPUTS, "af3"), os.path.join(DATA, "af3")]
RANK = re.compile(r"(?:model[_-]?(\d+)|rank[_-]?(\d+)|sample[_-]?(\d+))", re.I)


def rank_of(path):
    m = RANK.search(os.path.basename(path))
    if not m:
        return 99
    return int(next(g for g in m.groups() if g is not None))


def find_model(job):
    hits = []
    for root in ROOTS:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            for f in filenames:
                if not f.lower().endswith((".cif", ".pdb")):
                    continue
                rel = os.path.relpath(os.path.join(dirpath, f), root).lower()
                if job.lower().replace("__", "_") in rel.replace("__", "_"):
                    hits.append(os.path.join(dirpath, f))
    if not hits:
        seen = sorted({os.path.relpath(d, r) for r in ROOTS if os.path.isdir(r)
                       for d, _sub, fs in os.walk(r)
                       if any(f.lower().endswith((".cif", ".pdb")) for f in fs)})
        fatal("no .cif or .pdb carries the job name %s. Roots searched: %s. Folders "
              "holding a .cif or .pdb below them: %s. Either unpack the AlphaFold3 "
              "results below one of those roots keeping the job name in the path, or "
              "set MODELS_ROOT to wherever they are."
              % (job, ROOTS, seen or "none"))
    hits.sort(key=lambda p: (rank_of(p), p))
    best = hits[0]
    if len(hits) > 1:
        print("   %d files match %s; taking %s, which the ranking names first"
              % (len(hits), job, os.path.basename(best)))
    return best


def protein_chains(st):
    out = []
    for ch in st[0]:
        n = 0
        for r in ch:
            info = gemmi.find_tabulated_residue(r.name)
            if info is not None and info.is_amino_acid():
                n += 1
        if n:
            out.append((ch.name, n))
    return out


def load(path):
    st = gemmi.read_structure(path)
    st.setup_entities()
    st.remove_alternative_conformations()
    st.remove_hydrogens()
    st.remove_waters()
    return st


ref_path = None
for ext in (".cif", ".pdb"):
    cand = os.path.join(STRUCTURES, REF_PDB + ext)
    if os.path.isfile(cand):
        ref_path = cand
        break
if ref_path is None:
    fatal("no %s.cif or %s.pdb in %s. The project's fetch and check script puts the "
          "deposited structures there, so run that first." % (REF_PDB, REF_PDB, STRUCTURES))
REF_ST = load(ref_path)
ref_cof = None
for ch in REF_ST[0]:
    for r in ch:
        if r.name == COFACTOR and ref_cof is None:
            ref_cof = (ch.name, r)
if ref_cof is None:
    fatal("%s holds no residue named %s, so there is no cofactor to copy"
          % (ref_path, COFACTOR))
ref_chain = ref_cof[0]
print("\ncofactor donor: %s chain %s, residue %s%d"
      % (os.path.basename(ref_path), ref_chain, ref_cof[1].name, ref_cof[1].seqid.num))

C4A = ref_cof[1].find_atom("C4A", "*")
if C4A is None:
    fatal("%s %s carries no C4A, so the aldimine cannot be located. A cofactor "
          "modelled without its C4A is not the internal aldimine."
          % (os.path.basename(ref_path), COFACTOR))
near = []
for ch in REF_ST[0]:
    for r in ch:
        nz = r.find_atom("NZ", "*")
        if nz is not None:
            near.append((C4A.pos.dist(nz.pos), ch.name, r.name, r.seqid.num))
near.sort()
if not near:
    fatal("%s holds no NZ atom at all, so there is no lysine for the cofactor to be "
          "bonded to" % os.path.basename(ref_path))
bond_A, lys_chain, lys_name, SCHIFF_BASE_RESIDUE = near[0]
if bond_A > MAX_ALDIMINE_A:
    fatal("the nearest NZ to the cofactor's C4A is %s%d at %.2f A, beyond the %.1f A "
          "an aldimine allows. This build substitutes a lysine that carries the "
          "cofactor, and %s does not model one."
          % (lys_name, SCHIFF_BASE_RESIDUE, bond_A, MAX_ALDIMINE_A, REF_PDB))
if lys_name != "LYS":
    fatal("the residue bonded to the cofactor is %s%d, not a lysine"
          % (lys_name, SCHIFF_BASE_RESIDUE))
if len(near) > 1 and near[1][0] <= MAX_ALDIMINE_A:
    fatal("two lysines sit within %.1f A of the cofactor's C4A, %s%d at %.2f A and "
          "%s%d at %.2f A, so which one carries it is ambiguous"
          % (MAX_ALDIMINE_A, near[0][2], near[0][3], near[0][0],
             near[1][2], near[1][3], near[1][0]))
print("internal aldimine: %s C4A to %s%d NZ, %.2f A. CHARMM-GUI substitutes that "
      "lysine." % (COFACTOR, lys_name, SCHIFF_BASE_RESIDUE, bond_A))

LYS_ATOMS = sorted({"N", "HN", "H", "HT1", "HT2", "HT3", "CA", "HA", "CB", "HB1",
                    "HB2", "HB3", "CG", "HG1", "HG2", "HG3", "CD", "HD1", "HD2",
                    "HD3", "CE", "HE1", "HE2", "HE3", "NZ", "HZ1", "HZ2", "HZ3",
                    "C", "O", "OT1", "OT2", "OXT"})
BUILT_RESIDUE = ["LLPP", "LLP"]
MOIETY_HEAVY_MIN = sum(1 for a in ref_cof[1] if a.element.name != "H") - 1


def ca_map(chain):
    out = {}
    for r in chain:
        a = r.find_atom("CA", "*")
        if a is not None:
            out[r.seqid.num] = a.pos
    return out


def place_cofactor(model_st, target_chain_name, copy_index):
    donor = REF_ST[0][ref_chain]
    target = model_st[0][target_chain_name]
    dmap, tmap = ca_map(donor), ca_map(target)
    shared = sorted(set(dmap) & set(tmap) & set(range(1, RETAINED_END + 1)))
    if len(shared) < 50:
        fatal("the crystal subunit and the model chain share only %d alpha carbons in "
              "1 to %d, which is too few to superpose on" % (len(shared), RETAINED_END))
    sup = gemmi.superpose_positions([tmap[i] for i in shared],
                                    [dmap[i] for i in shared])
    if sup.rmsd > MAX_SUPERPOSE_RMSD_A:
        fatal("the crystal subunit fits the model chain at %.2f A RMSD over %d alpha "
              "carbons of the retained block, above the %.1f A allowed. The cofactor "
              "pose would be placed from a bad fit, which is worse than leaving it "
              "out." % (sup.rmsd, len(shared), MAX_SUPERPOSE_RMSD_A))
    new = gemmi.Residue()
    new.name = COFACTOR
    new.seqid = gemmi.SeqId(9000 + copy_index, " ")
    new.het_flag = "H"
    for atom in ref_cof[1]:
        a = gemmi.Atom()
        a.name, a.element, a.charge = atom.name, atom.element, atom.charge
        a.occ, a.b_iso = 1.0, atom.b_iso
        a.pos = gemmi.Position(sup.transform.apply(atom.pos))
        new.add_atom(a)
    target.add_residue(new)
    return sup.rmsd, len(shared), len(new)


written, summary = [], {}
for build in BUILD_ORDER:
    job = BUILD_JOB[build]
    print("\n" + "=" * 72)
    print("BUILD %s   from job %s" % (build.upper(), job))
    print("=" * 72)
    path = find_model(job)
    print("   model: %s" % path)
    st = load(path)
    chains = protein_chains(st)
    if len(chains) != 2:
        fatal("%s holds %d protein chains %s and this build needs two"
              % (path, len(chains), chains))
    got = sorted(n for _, n in chains)
    want = sorted(CHAIN_LENGTHS[build])
    if got != want:
        fatal("%s holds chains of %s residues and the %s build needs %s. The job name "
              "in the path may not be the job that produced it." % (path, got, build, want))
    present = [(ch.name, r.seqid.num) for ch in st[0] for r in ch if r.name == COFACTOR]
    if present:
        print("   the model already carries %d copies of %s; they are removed and the "
              "crystal pose is used instead, so both builds carry the same pose"
              % (len(present), COFACTOR))
        for ch in st[0]:
            for i in range(len(ch) - 1, -1, -1):
                if ch[i].name == COFACTOR:
                    del ch[i]
    order = sorted(chains, key=lambda c: -c[1])
    hosts = [order[0][0]] if COFACTOR_COPIES[build] == 1 else [c[0] for c in order]
    if len(hosts) != COFACTOR_COPIES[build]:
        fatal("the %s build asks for %d cofactor copies and %d host chains were "
              "resolved" % (build, COFACTOR_COPIES[build], len(hosts)))
    placed = []
    for k, host in enumerate(hosts, 1):
        rmsd, n_ca, n_at = place_cofactor(st, host, k)
        placed.append(dict(chain=host, rmsd_A=round(rmsd, 3), alpha_carbons=n_ca,
                           atoms=n_at))
        print("   cofactor %d placed on chain %s, fit %.2f A over %d alpha carbons, "
              "%d atoms" % (k, host, rmsd, n_ca, n_at))
    st.setup_entities()
    out = os.path.join(CHARMM, "%s.pdb" % build)
    st.write_pdb(out)
    back = gemmi.read_structure(out)
    n_cof = sum(1 for ch in back[0] for r in ch if r.name == COFACTOR)
    if n_cof != COFACTOR_COPIES[build]:
        fatal("%s was written with %d copies of %s and should hold %d"
              % (out, n_cof, COFACTOR, COFACTOR_COPIES[build]))
    print("   wrote %s, %d chains, %d copies of %s"
          % (out, len(back[0]), n_cof, COFACTOR))
    written.append(out)
    summary[build] = dict(model=path, chains=[list(c) for c in chains],
                          cofactor_placements=placed, pdb=out)

SUBSTITUTIONS = {}
for build in BUILD_ORDER:
    order = [c[0] for c in summary[build]["chains"]]
    rows = []
    for place in summary[build]["cofactor_placements"]:
        if place["chain"] not in order:
            fatal("the cofactor was placed on chain %s and the written file holds %s"
                  % (place["chain"], order))
        rows.append("PRO" + "ABCDEFGH"[order.index(place["chain"])])
    SUBSTITUTIONS[build] = rows

SPEC = dict(
    accession="Q16719", isoform="Q16719-2",
    builds=BUILD_ORDER, build_job=BUILD_JOB,
    chain_lengths=CHAIN_LENGTHS, cofactor=COFACTOR,
    cofactor_copies=COFACTOR_COPIES,
    schiff_base_residue=SCHIFF_BASE_RESIDUE, aldimine_A=round(bond_A, 2),
    substitutions=SUBSTITUTIONS,
    built_residue=BUILT_RESIDUE, lysine_atoms=LYS_ATOMS,
    moiety_heavy_min=MOIETY_HEAVY_MIN,
    retained_end=RETAINED_END, lost_block=[LOST_FROM, LOST_TO],
    crystal_partner_burial_A2=CRYSTAL, cofactor_source=REF_PDB,
    max_superpose_rmsd_A=MAX_SUPERPOSE_RMSD_A,
    ref_t_K=REF_T_K, inputs=summary,
    written=datetime.datetime.now().isoformat(timespec="seconds"))
spec_path = os.path.join(INPUTS, "build_spec.json")
json.dump(SPEC, open(spec_path, "w"), indent=2)
print("\nwrote %s" % spec_path)

print("\n" + "=" * 72)
print("CHARMM-GUI checklist, the same settings for both builds")
print("=" * 72)
print("""
Solution Builder, once per build, uploading the PDB written above.

  1  Upload %s, then %s. PDB format, not PDB ID.

  2  Model and chain selection. Keep both protein chains. UNTICK the %s hetero
     rows. The cofactor is not a separate molecule here: both deposited human
     structures model it as the internal aldimine, a covalent bond from its C4A to
     %s%d NZ, measured at %.2f A in %s. CHARMM-GUI builds that as one non-standard
     residue in place of the lysine, so a hetero %s left ticked would put a second,
     unbonded copy into the system. Leave Check pKa off.

  3  On the PDB manipulation page, under Non-standard amino acid substitution, add
     %d row%s:
%s
     LLPP is the merged lysine and cofactor with one negative charge on the
     phosphate. LLP is the same residue with a neutral phosphate. Neither is the
     dianion, so the model carries one proton more on the phosphate than the species
     that dominates at pH 7; the choice is the same in both builds, so it cannot
     produce a difference between them.

     Press Generate PDB and read what it reports. It has to name as many LLPP
     residues as that build has rows above, and no missing atoms.

  4  Terminal groups: default. Protonation: default at pH 7.

  5  Water box: rectangular. Build canonical first with the box fitted to the protein
     at a 10 A edge distance, then read the box out of its download, from the last
     line of gromacs/step3_input.gro in nanometres or from SET A, SET B and SET C in
     step2.1_waterbox.prm in angstroms. Build mixed with those same dimensions typed
     into Specify Waterbox Size. The same box for both keeps the solvent identical,
     so the only difference between the two systems is the partner chain. A box
     fitted separately to each would make the smaller system smaller again in its
     water, and the atom counts would then differ by more than the truncation
     accounts for, which part 2 refuses.

  6  Ions: KCl at 0.15 M, neutralising, Monte-Carlo placement. The two builds carry
     different net charge because one chain is shorter, so the ion counts will
     differ, which is expected. The counts are also a check on step 3: each LLPP
     replaces a lysine (+1) with a neutral Schiff base nitrogen and a phosphate
     monoanion (-1), so every substitution moves the solute charge by -2, and the
     excess of potassium over chloride should show it.

  7  Force field: CHARMM36m, TIP3P water.

  8  Dynamics: NPT, temperature %g K, and GROMACS as the output format.

  9  Download the whole gromacs/ folder, toppar included, and unpack it on Drive
     under a folder whose name carries the word canonical or mixed.

Part 2 reads the temperature, the chain lengths and the cofactor back from the
download, and superposes the file that was uploaded here onto what came back to see
where CHARMM-GUI put the cofactor. A setting typed wrongly, or a ring built into
solvent, is caught before a chunk is run rather than after.
""" % (os.path.basename(written[0]), os.path.basename(written[1]), COFACTOR,
       lys_name, SCHIFF_BASE_RESIDUE, bond_A, REF_PDB, COFACTOR,
       sum(len(v) for v in SUBSTITUTIONS.values()),
       "s in all, split across the two builds as shown",
       "\n".join("        LLPP   segment %-6s residue %-5d  (%s build)"
                 % (seg, SCHIFF_BASE_RESIDUE, build)
                 for build in BUILD_ORDER for seg in SUBSTITUTIONS[build]),
       REF_T_K))
